In [1]:
import pandas as pd

strong_df = pd.read_csv("data/strong_export.csv")
hevy_df = pd.read_csv("data/hevy_export.csv")

strong_df['date'] = pd.to_datetime(strong_df['Date'], format='%Y-%m-%d %H:%M:%S')
hevy_df['date'] = pd.to_datetime(hevy_df['start_time'], format='%d %b %Y, %H:%M')

In [2]:
strong_df['Exercise Name'] = strong_df['Exercise Name'].replace('Squat (Band)', 'Squat (Barbell)')

HEVY_TO_CANONICAL = {
    'Chest Fly (Machine)': 'Pec Deck (Machine)',
    'Rear Delt Reverse Fly (Machine)': 'Reverse Fly (Machine)',
    'Triceps Pushdown': 'Triceps Pushdown (Cable - Straight Bar)',
    'Seated Calf Raise': 'Seated Calf Raise (Machine)',
    'Shoulder Press (Dumbbell)': 'Seated Overhead Press (Dumbbell)',
    'Overhead Triceps Extension (Cable)': 'Triceps Extension (Cable)',
    'Hanging Leg Raise': 'Leg Raise Parallel Bars',
}
hevy_df['exercise_title'] = hevy_df['exercise_title'].replace(HEVY_TO_CANONICAL)

In [3]:
strong_clean = strong_df.rename(columns={
    'Exercise Name': 'exercise',
    'Weight': 'weight',
    'Reps': 'reps',
    'Set Order': 'set_order',
})[['date', 'exercise', 'weight', 'reps', 'set_order']]
strong_clean['source'] = 'strong'

hevy_clean = hevy_df.rename(columns={
    'exercise_title': 'exercise',
    'weight_lbs': 'weight',
    'reps': 'reps',
    'set_index': 'set_order',
})[['date', 'exercise', 'weight', 'reps', 'set_order']]
hevy_clean['source'] = 'hevy'

In [4]:
combined = pd.concat([strong_clean, hevy_clean], ignore_index=True)
combined = combined.sort_values('date').reset_index(drop=True)
combined

,date,exercise,weight,reps,set_order,source
0,2025-09-19 10:30:11,Squat (Barbell),135.0,8.0,1,strong
1,2025-09-19 10:30:11,Lat Pulldown (Cable),130.0,7.0,2,strong
2,2025-09-19 10:30:11,Bench Press (Barbell),95.0,13.0,2,strong
3,2025-09-19 10:30:11,Bench Press (Barbell),95.0,14.0,1,strong
4,2025-09-19 10:30:11,Seated Leg Curl (Machine),130.0,10.0,3,strong
...,...,...,...,...,...,...
1223,2026-07-04 13:24:00,Seated Row (Machine),145.0,6.0,0,hevy
1224,2026-07-04 13:24:00,Pull Up,NaN,5.0,1,hevy
1225,2026-07-04 13:24:00,Pull Up,NaN,5.0,0,hevy
1226,2026-07-04 13:24:00,Preacher Curl (Machine),110.0,6.0,1,hevy


In [5]:
BODYWEIGHT_EXERCISES = ['Pull Up', 'Leg Raise Parallel Bars']

modeling_df = combined[~combined['exercise'].isin(BODYWEIGHT_EXERCISES)].copy()

print("combined rows:", len(combined))
print("modeling_df rows:", len(modeling_df))
print("Any bodyweight exercises left?", modeling_df['exercise'].isin(BODYWEIGHT_EXERCISES).any())

combined rows: 1228
modeling_df rows: 1209
Any bodyweight exercises left? False


In [6]:
sorted_df = modeling_df.sort_values(['date', 'exercise', 'weight', 'reps'])
e1rm_df = sorted_df.drop_duplicates(subset=['date', 'exercise'], keep='last').copy()

print("modeling_df rows:", len(modeling_df))
print("e1rm_df rows:", len(e1rm_df))
e1rm_df.head(10)

modeling_df rows: 1209
e1rm_df rows: 434


,date,exercise,weight,reps,set_order,source
3,2025-09-19 10:30:11,Bench Press (Barbell),95.0,14.0,1,strong
6,2025-09-19 10:30:11,Lat Pulldown (Cable),130.0,9.0,1,strong
9,2025-09-19 10:30:11,Seated Calf Raise (Machine),70.0,11.0,1,strong
5,2025-09-19 10:30:11,Seated Leg Curl (Machine),130.0,12.0,2,strong
11,2025-09-19 10:30:11,Squat (Barbell),145.0,6.0,2,strong
31,2025-09-22 14:15:46,Bench Press (Barbell),125.0,5.0,2,strong
27,2025-09-22 14:15:46,Incline Bench Press (Dumbbell),40.0,6.0,3,strong
19,2025-09-22 14:15:46,Lateral Raise (Dumbbell),20.0,9.0,1,strong
13,2025-09-22 14:15:46,Pec Deck (Machine),135.0,8.0,1,strong
24,2025-09-22 14:15:46,Seated Overhead Press (Dumbbell),35.0,7.0,3,strong


In [7]:
e1rm_df['e1rm'] = (e1rm_df['weight'] * (1 + e1rm_df['reps'] / 30)).round(2)
e1rm_df.head(10)

,date,exercise,weight,reps,set_order,source,e1rm
3,2025-09-19 10:30:11,Bench Press (Barbell),95.0,14.0,1,strong,139.33
6,2025-09-19 10:30:11,Lat Pulldown (Cable),130.0,9.0,1,strong,169.00
9,2025-09-19 10:30:11,Seated Calf Raise (Machine),70.0,11.0,1,strong,95.67
5,2025-09-19 10:30:11,Seated Leg Curl (Machine),130.0,12.0,2,strong,182.00
11,2025-09-19 10:30:11,Squat (Barbell),145.0,6.0,2,strong,174.00
31,2025-09-22 14:15:46,Bench Press (Barbell),125.0,5.0,2,strong,145.83
27,2025-09-22 14:15:46,Incline Bench Press (Dumbbell),40.0,6.0,3,strong,48.00
19,2025-09-22 14:15:46,Lateral Raise (Dumbbell),20.0,9.0,1,strong,26.00
13,2025-09-22 14:15:46,Pec Deck (Machine),135.0,8.0,1,strong,171.00
24,2025-09-22 14:15:46,Seated Overhead Press (Dumbbell),35.0,7.0,3,strong,43.17


In [8]:
rolling = (
    e1rm_df.groupby('exercise')
    .rolling('28D', on='date')['e1rm']
    .mean()
    .reset_index()
    .rename(columns={'e1rm': 'rolling_e1rm'})
)

e1rm_df = e1rm_df.merge(rolling[['exercise', 'date', 'rolling_e1rm']], on=['exercise', 'date'], how='left')

e1rm_df.head(10)

,date,exercise,weight,reps,set_order,source,e1rm,rolling_e1rm
0,2025-09-19 10:30:11,Bench Press (Barbell),95.0,14.0,1,strong,139.33,139.33
1,2025-09-19 10:30:11,Lat Pulldown (Cable),130.0,9.0,1,strong,169.00,169.00
2,2025-09-19 10:30:11,Seated Calf Raise (Machine),70.0,11.0,1,strong,95.67,95.67
3,2025-09-19 10:30:11,Seated Leg Curl (Machine),130.0,12.0,2,strong,182.00,182.00
4,2025-09-19 10:30:11,Squat (Barbell),145.0,6.0,2,strong,174.00,174.00
5,2025-09-22 14:15:46,Bench Press (Barbell),125.0,5.0,2,strong,145.83,142.58
6,2025-09-22 14:15:46,Incline Bench Press (Dumbbell),40.0,6.0,3,strong,48.00,48.00
7,2025-09-22 14:15:46,Lateral Raise (Dumbbell),20.0,9.0,1,strong,26.00,26.00
8,2025-09-22 14:15:46,Pec Deck (Machine),135.0,8.0,1,strong,171.00,171.00
9,2025-09-22 14:15:46,Seated Overhead Press (Dumbbell),35.0,7.0,3,strong,43.17,43.17


In [9]:
e1rm_df['intensity_trend'] = (e1rm_df['weight'] / e1rm_df['rolling_e1rm']).round(4)
e1rm_df.head(10)

,date,exercise,weight,reps,set_order,source,e1rm,rolling_e1rm,intensity_trend
0,2025-09-19 10:30:11,Bench Press (Barbell),95.0,14.0,1,strong,139.33,139.33,0.6818
1,2025-09-19 10:30:11,Lat Pulldown (Cable),130.0,9.0,1,strong,169.00,169.00,0.7692
2,2025-09-19 10:30:11,Seated Calf Raise (Machine),70.0,11.0,1,strong,95.67,95.67,0.7317
3,2025-09-19 10:30:11,Seated Leg Curl (Machine),130.0,12.0,2,strong,182.00,182.00,0.7143
4,2025-09-19 10:30:11,Squat (Barbell),145.0,6.0,2,strong,174.00,174.00,0.8333
5,2025-09-22 14:15:46,Bench Press (Barbell),125.0,5.0,2,strong,145.83,142.58,0.8767
6,2025-09-22 14:15:46,Incline Bench Press (Dumbbell),40.0,6.0,3,strong,48.00,48.00,0.8333
7,2025-09-22 14:15:46,Lateral Raise (Dumbbell),20.0,9.0,1,strong,26.00,26.00,0.7692
8,2025-09-22 14:15:46,Pec Deck (Machine),135.0,8.0,1,strong,171.00,171.00,0.7895
9,2025-09-22 14:15:46,Seated Overhead Press (Dumbbell),35.0,7.0,3,strong,43.17,43.17,0.8107


In [10]:
squat_personal = e1rm_df[e1rm_df['exercise'] == 'Squat (Barbell)'].copy()
squat_personal = squat_personal.sort_values('date')
squat_personal['next_e1rm'] = squat_personal['e1rm'].shift(-1)
squat_personal = squat_personal.dropna(subset=['next_e1rm'])

squat_personal[['date', 'e1rm', 'rolling_e1rm', 'intensity_trend', 'next_e1rm']]

,date,e1rm,rolling_e1rm,intensity_trend,next_e1rm
4,2025-09-19 10:30:11,174.00,174.000000,0.8333,196.33
24,2025-09-26 10:37:57,196.33,185.165000,0.8371,196.33
63,2025-10-24 10:57:33,196.33,196.330000,0.7895,180.00
82,2025-10-31 10:12:50,180.00,188.165000,0.7175,186.00
113,2025-11-14 10:50:46,186.00,187.443333,0.8269,180.00
144,2025-12-13 13:54:33,180.00,180.000000,0.7500,180.00
161,2026-01-13 10:25:48,180.00,180.000000,0.7500,180.00
178,2026-01-27 10:11:30,180.00,180.000000,0.7500,180.00
209,2026-02-17 09:59:58,180.00,180.000000,0.7500,180.00
227,2026-03-26 10:25:11,180.00,180.000000,0.7500,196.33


In [11]:
from sklearn.model_selection import train_test_split
FEATURES = ['e1rm', 'rolling_e1rm', 'intensity_trend']
TARGET = 'next_e1rm'

X = squat_personal[FEATURES]
y = squat_personal[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train date range:", squat_personal['date'].iloc[:len(X_train)].min(), "to", squat_personal['date'].iloc[:len(X_train)].max())
print("Test date range:", squat_personal['date'].iloc[len(X_train):].min(), "to", squat_personal['date'].iloc[len(X_train):].max())

Training rows: 15
Test rows: 4
Train date range: 2025-09-19 10:30:11 to 2026-05-31 17:56:00
Test date range: 2026-06-07 14:53:00 to 2026-06-24 11:55:00


In [12]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

squat_ridge = Ridge()
squat_ridge.fit(X_train, y_train)

train_pred = squat_ridge.predict(X_train)
test_pred = squat_ridge.predict(X_test)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("Ridge — Train MAE:", round(train_mae, 2))
print("Ridge — Test MAE:", round(test_mae, 2))
print("Ridge — Test RMSE:", round(test_rmse, 2))

Ridge — Train MAE: 6.0
Ridge — Test MAE: 18.61
Ridge — Test RMSE: 19.97


In [13]:
persistence_pred = X_test['e1rm']  # naive guess: next session = same as current session
persistence_mae = mean_absolute_error(y_test, persistence_pred)

print("Persistence baseline MAE:", round(persistence_mae, 2))
print("Ridge Test MAE:", round(test_mae, 2))

Persistence baseline MAE: 8.83
Ridge Test MAE: 18.61


In [14]:
from xgboost import XGBRegressor

squat_xgb = XGBRegressor(random_state=42)
squat_xgb.fit(X_train, y_train)

train_pred_xgb = squat_xgb.predict(X_train)
test_pred_xgb = squat_xgb.predict(X_test)

train_mae_xgb = mean_absolute_error(y_train, train_pred_xgb)
test_mae_xgb = mean_absolute_error(y_test, test_pred_xgb)

print("XGBoost — Train MAE:", round(train_mae_xgb, 2))
print("XGBoost — Test MAE:", round(test_mae_xgb, 2))

XGBoost — Train MAE: 3.42
XGBoost — Test MAE: 20.68


In [15]:
def evaluate_personal_model(exercise_name):
    data = e1rm_df[e1rm_df['exercise'] == exercise_name].copy()
    data = data.sort_values('date')
    data['next_e1rm'] = data['e1rm'].shift(-1)
    data = data.dropna(subset=['next_e1rm'])

    FEATURES = ['e1rm', 'rolling_e1rm', 'intensity_trend']
    TARGET = 'next_e1rm'

    X = data[FEATURES]
    y = data[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    persistence_pred = X_test['e1rm']
    persistence_mae = mean_absolute_error(y_test, persistence_pred)

    ridge = Ridge()
    ridge.fit(X_train, y_train)
    ridge_train_mae = mean_absolute_error(y_train, ridge.predict(X_train))
    ridge_test_mae = mean_absolute_error(y_test, ridge.predict(X_test))

    xgb = XGBRegressor(random_state=42)
    xgb.fit(X_train, y_train)
    xgb_train_mae = mean_absolute_error(y_train, xgb.predict(X_train))
    xgb_test_mae = mean_absolute_error(y_test, xgb.predict(X_test))

    print(f"\n{exercise_name} (train={len(X_train)}, test={len(X_test)})")
    print(f"  Persistence MAE: {persistence_mae:.2f}")
    print(f"  Ridge   — Train: {ridge_train_mae:.2f}, Test: {ridge_test_mae:.2f}")
    print(f"  XGBoost — Train: {xgb_train_mae:.2f}, Test: {xgb_test_mae:.2f}")

evaluate_personal_model('Bench Press (Barbell)')


Bench Press (Barbell) (train=26, test=7)
  Persistence MAE: 2.83
  Ridge   — Train: 6.28, Test: 14.59
  XGBoost — Train: 0.73, Test: 17.75


In [16]:
import joblib
import pandas as pd

squat_population_model = joblib.load('squat_population_model.joblib')

my_profile = pd.DataFrame([{
    'Age': 19,
    'BodyweightKg': 74.8,
    'Sex_encoded': 0,
    'experience': 1
}])

my_baseline_squat = squat_population_model.predict(my_profile)[0]
print("Population baseline squat estimate for you:", round(my_baseline_squat, 2))

Population baseline squat estimate for you: 173.54


In [17]:
def check_plateau(exercise_name, threshold=0.02, lookback_sessions=4):
    data = e1rm_df[e1rm_df['exercise'] == exercise_name].sort_values('date')

    if len(data) < lookback_sessions + 1:
        print(f"{exercise_name}: not enough sessions to assess yet")
        return None

    recent_rolling = data['rolling_e1rm'].iloc[-1]
    past_rolling = data['rolling_e1rm'].iloc[-(lookback_sessions + 1)]
    percent_change = (recent_rolling - past_rolling) / past_rolling

    is_plateau = percent_change < threshold

    print(f"{exercise_name}: rolling e1RM {lookback_sessions} sessions ago = {past_rolling:.2f}, now = {recent_rolling:.2f} ({percent_change*100:.2f}% change)")
    print("Plateau flag:", is_plateau)
    return is_plateau

check_plateau('Squat (Barbell)')
check_plateau('Bench Press (Barbell)')

Squat (Barbell): rolling e1RM 4 sessions ago = 198.61, now = 213.23 (7.36% change)
Plateau flag: False
Bench Press (Barbell): rolling e1RM 4 sessions ago = 161.42, now = 170.12 (5.39% change)
Plateau flag: False


np.False_

In [18]:
def bucket_experience(exp):
    if exp <= 5:
        return '1-5'
    elif exp <= 10:
        return '6-10'
    elif exp <= 20:
        return '11-20'
    elif exp <= 40:
        return '21-40'
    else:
        return '41+'


In [19]:
def evaluate_shrinkage_model(exercise_name, population_rates, k=15):
    data = e1rm_df[e1rm_df['exercise'] == exercise_name].copy()
    data = data.sort_values('date').reset_index(drop=True)
    data['next_e1rm'] = data['e1rm'].shift(-1)
    data['session_number'] = data.index + 1

    data['trend_kg'] = data['e1rm'].diff(4)
    data['trend_days'] = (data['date'] - data['date'].shift(4)).dt.days
    data['personal_rate_per_day'] = data['trend_kg'] / data['trend_days']

    data['experience_bucket'] = data['session_number'].apply(bucket_experience)
    data['population_rate'] = data['experience_bucket'].map(population_rates)

    data['days_to_next'] = (data['date'].shift(-1) - data['date']).dt.days

    data = data.dropna(subset=['next_e1rm', 'personal_rate_per_day', 'population_rate', 'days_to_next'])

    data['w'] = data['session_number'] / (data['session_number'] + k)
    data['blended_rate'] = data['w'] * data['personal_rate_per_day'] + (1 - data['w']) * data['population_rate']

    data['shrunk_pred'] = data['e1rm'] + data['blended_rate'] * data['days_to_next']
    data['persistence_pred'] = data['e1rm']

    shrunk_mae = mean_absolute_error(data['next_e1rm'], data['shrunk_pred'])
    persistence_mae = mean_absolute_error(data['next_e1rm'], data['persistence_pred'])

    print(f"{exercise_name} (n={len(data)})")
    print(f"  Persistence MAE:             {persistence_mae:.2f}")
    print(f"  Population-shrunk trend MAE: {shrunk_mae:.2f}")

    return data

evaluate_shrinkage_model('Squat (Barbell)', squat_population_rates)


NameError: name 'squat_population_rates' is not defined

In [ ]:
squat_population_rates = {'1-5': 0.032468, '6-10': 0.017094, '11-20': 0.010373, '21-40': 0.003497, '41+': 0.000000}
squat_population_var = {'1-5': 1.289652, '6-10': 2.054512, '11-20': 8.526419, '21-40': 8.663513, '41+': 0.126471}

bench_population_rates = {'1-5': 0.016190, '6-10': 0.007215, '11-20': 0.000000, '21-40': 0.000000, '41+': 0.000000}
bench_population_var = {'1-5': 0.532288, '6-10': 1.583747, '11-20': 3.245305, '21-40': 10.906004, '41+': 9.456615}

def evaluate_shrinkage_model_v2(exercise_name, population_rates, population_var):
    data = e1rm_df[e1rm_df['exercise'] == exercise_name].copy()
    data = data.sort_values('date').reset_index(drop=True)
    data['next_e1rm'] = data['e1rm'].shift(-1)
    data['session_number'] = data.index + 1

    data['trend_kg'] = data['e1rm'].diff(4)
    data['trend_days'] = (data['date'] - data['date'].shift(4)).dt.days
    data['personal_rate_per_day'] = data['trend_kg'] / data['trend_days']

    data['experience_bucket'] = data['session_number'].apply(bucket_experience)
    data['population_rate'] = data['experience_bucket'].map(population_rates)
    data['population_variance'] = data['experience_bucket'].map(population_var)

    data['days_to_next'] = (data['date'].shift(-1) - data['date']).dt.days

    # measured noise in YOUR OWN rate estimate, from your own history
    personal_var = data['personal_rate_per_day'].var()

    data = data.dropna(subset=['next_e1rm', 'personal_rate_per_day', 'population_rate', 'population_variance', 'days_to_next'])

    # empirical-Bayes weight: trust personal more as your session count grows,
    # trust population more when population lifters cluster tightly around its median
    data['w'] = data['population_variance'] / (data['population_variance'] + (personal_var / data['session_number']))
    data['w'] = data['w'].clip(0, 1)

    data['blended_rate'] = data['w'] * data['personal_rate_per_day'] + (1 - data['w']) * data['population_rate']

    # anchor on the smoothed rolling e1RM instead of one raw session, to reduce starting-point noise too
    data['shrunk_pred'] = data['rolling_e1rm'] + data['blended_rate'] * data['days_to_next']
    data['persistence_pred'] = data['e1rm']

    shrunk_mae = mean_absolute_error(data['next_e1rm'], data['shrunk_pred'])
    persistence_mae = mean_absolute_error(data['next_e1rm'], data['persistence_pred'])

    print(f"{exercise_name} (n={len(data)})")
    print(f"  Persistence MAE:            {persistence_mae:.2f}")
    print(f"  Population-shrunk v2 MAE:   {shrunk_mae:.2f}")

    return data

evaluate_shrinkage_model_v2('Squat (Barbell)', squat_population_rates, squat_population_var)
evaluate_shrinkage_model_v2('Bench Press (Barbell)', bench_population_rates, bench_population_var)

Squat (Barbell) (n=15)
  Persistence MAE:            4.42
  Population-shrunk v2 MAE:   6.52
Bench Press (Barbell) (n=29)
  Persistence MAE:            7.47
  Population-shrunk v2 MAE:   8.02


,date,exercise,weight,reps,set_order,source,e1rm,rolling_e1rm,intensity_trend,next_e1rm,...,trend_days,personal_rate_per_day,experience_bucket,population_rate,population_variance,days_to_next,w,blended_rate,shrunk_pred,persistence_pred
4,2025-10-06 13:32:42,Bench Press (Barbell),125.0,6.0,1,strong,150.00,144.032000,0.8679,154.17,...,17.0,0.627647,1-5,0.016190,0.532288,14.0,0.927642,0.583403,152.199642,150.00
5,2025-10-20 13:56:58,Bench Press (Barbell),125.0,7.0,1,strong,154.17,147.000000,0.8503,135.00,...,27.0,0.308889,6-10,0.007215,1.583747,3.0,0.978620,0.302439,147.907318,154.17
6,2025-10-24 10:57:33,Bench Press (Barbell),90.0,15.0,1,strong,135.00,147.292500,0.6110,150.00,...,28.0,0.000000,6-10,0.007215,1.583747,3.0,0.981618,0.000133,147.292898,135.00
7,2025-10-27 13:47:30,Bench Press (Barbell),125.0,6.0,1,strong,150.00,147.834000,0.8455,135.00,...,27.0,0.000000,6-10,0.007215,1.583747,3.0,0.983879,0.000116,147.834349,150.00
8,2025-10-31 10:12:50,Bench Press (Barbell),90.0,15.0,1,strong,135.00,144.834000,0.6214,145.83,...,24.0,-0.625000,6-10,0.007215,1.583747,3.0,0.985645,-0.615924,142.986227,135.00
9,2025-11-03 13:54:46,Bench Press (Barbell),125.0,5.0,1,strong,145.83,144.000000,0.8681,150.00,...,13.0,-0.641538,6-10,0.007215,1.583747,7.0,0.987061,-0.633145,139.567988,145.83
10,2025-11-10 14:26:32,Bench Press (Barbell),125.0,6.0,1,strong,150.00,145.000000,0.8621,135.00,...,17.0,0.882353,11-20,0.000000,3.245305,3.0,0.994218,0.877251,147.631754,150.00
11,2025-11-14 10:50:46,Bench Press (Barbell),90.0,15.0,1,strong,135.00,143.571429,0.6269,151.67,...,17.0,-0.882353,11-20,0.000000,3.245305,3.0,0.994698,-0.877674,140.938406,135.00
12,2025-11-17 13:41:34,Bench Press (Barbell),130.0,5.0,2,strong,151.67,144.583750,0.8991,144.00,...,17.0,0.980588,11-20,0.000000,3.245305,20.0,0.995103,0.975787,164.099484,151.67
13,2025-12-08 10:25:18,Bench Press (Barbell),135.0,2.0,2,strong,144.00,145.167500,0.9300,135.00,...,34.0,-0.053824,11-20,0.000000,3.245305,5.0,0.995452,-0.053579,144.899606,144.00
